# Measure model perfomance with ROC 
Usecase example: only top confident predictions are important 

This script allows to visualize the accuracy of a model's predictions across different percentiles of predicted probabilities. 


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
np.set_printoptions(suppress=True)
from decimal import Decimal

# Fake Data to test

In [ ]:

# Generate synthetic data
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
x = np.arange(0.05, 1.05, 0.1)
classes_acc = {round(i,2):[] for i in np.arange(0.05, 1.05, 0.1)}
classes_acc

In [ ]:
def probs_to_percentiles(y_probs: list, min_samples: int, x=np.arange(0.05, 1.05, 0.1)):
    ''' Put probabily in its percentile  
    10 Percentiles from 0:1 with step = 0.1
    X saves the middle of each percentile, important for plots
    '''
    classes_acc = {round(i,2):[] for i in x}
    mapping = {v: round(x[i],2) for i, v in enumerate(range(0, 10))}
    for i, prob in enumerate(y_probs):
        # correct floating point
        percentile = min(int(Decimal(str(prob)) // Decimal('0.1')), 9)
        # append 1, 0 based on if y_test was inded == 1
        # since we are always assuming 1 as the choise
        classes_acc[mapping[percentile]].append(y_test[i])
        
    _ = [print(i, len(prob_class)) for i, prob_class in classes_acc.items()]
    return {i: sum(prob_class)/len(prob_class) 
            for i, prob_class in classes_acc.items()
            if len(prob_class) > min_samples}
    

In [ ]:
def plot_accuracy_by_percentile(y_probs: list, accuracys: dict):
    ''' Plot Percentile Group and its Accuracy 
    Plot Predicted Probabilities 
    
    :param @y_probs --> model predicted probabilities
    :param @accuracys --> dict of percentile(middle of range): accuracy
    '''
    # accuracy points and probabilities of X_test
    plt.scatter(list(accuracys.keys()), 
                list(accuracys.values()), color='red')
    plt.plot(np.linspace(0, 1, len(y_probs)), 
             sorted(y_probs), color='blue')

    # Reverse the x-axis
    plt.gca().invert_xaxis()
    # Set x-ticks as x-values
    plt.xticks(np.arange(0.0, 1.1, 0.1))

    # Add labels and title
    plt.ylabel('Probabilities')
    #plt.title('Reversed X-Axis with Vertical Lines')

    # Show the plot
    plt.show()

# Result 

Red scatter points, are the percentiles as the x-coordinates and the accuracy scores as the y-coordinates. It then plots the sorted predicted 
The Blue line, correspond to the probabilities of model predictions 

In [ ]:
y_probs = model.predict_proba(X_test)[:, 1]
accuracys = probs_to_percentiles(y_probs, min_samples=5)
plot_accuracy_by_percentile(y_probs, accuracys)